# 38 · Regression, one model per lag, checked against pandas and statsmodels

## What this checks

**Regression** can fit one model per lag in a single step. This check covers release 0.9.0, and that feature.

A lag asks about the weather some time before the cases, or after. **Relate** brings in a column once for each
shift in time. Regression then fits the same model once for each shifted column, each on its own rows. An adjustment
can follow the same shift: the temperature 14 days before is adjusted for the rain 14 days before.

This used to take a Relate and a Regression for every lag. One of each now does it. If a model used the wrong rows,
or the wrong shift, its lag would answer a different question.

## How it is checked

We build each lag and fit each model again, sharing no code with Fieldwork.

- **pandas**, Python's library for tables, makes each lagged column itself. It matches each week with the same
  municipality's week that starts that many days later.
- **statsmodels**, a Python library for statistical models, fits each lag on that lag's complete rows. It adds one
  yes-or-no column per stratum, a group of weeks compared only with each other.

Neither reads Fieldwork's shifted table.

The data are those of check 37: two municipalities of Rio de Janeiro state, 2010 to 2025, without the twelve weeks
that appear twice at the turn of a year. Each model is weekly cases on the weekly air temperature, adjusted for the
weekly rainfall at the same shift. The shifts are 0, 14 and 28 days before, and 14 days after. Weeks are compared
within a municipality, year and month.

## What should happen

1. **Each model rests on its own rows.** Its rows used, strata used and strata dropped equal pandas' counts for that
   lag's columns alone. A model on the rows complete at every lag would use fewer rows, and the data's are not those.
2. **Each coefficient** is within one part in a million of statsmodels'.
3. **Each standard error**, the measure of a coefficient's uncertainty, is within one part in a hundred thousand.
   It is widened by the Pearson chi-square over the residual degrees of freedom, as in check 37. This allows for
   counts that vary more than a plain Poisson model, a regression for counts, expects.
4. **The table names each model's shift.** Its columns include the shift in days, and each model's shift is the one
   declared.

These criteria were fixed before the check was run.

In [ ]:
# Only what this notebook imports. See 00-kernel-check for what this kernel really has.
NEEDED = ["pandas", "statsmodels"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested:", NEEDED)
except ImportError:
    print("Not Pyodide; using the environment as installed.")

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_38_lag_models import run

report = run(Path.cwd() / "fixtures" / "lag-models-brazil-001.json")
print("agrees:", report["agrees"])
for c in report["comparisons"]:
    print(f"\n{c['column']} with {c['rain']}: rows {c['counts']['rowsUsed']} (complete at every lag: {c['rowsCompleteAtEveryLag']}),"
          f" strata {c['counts']['strataUsed']}, dropped {c['counts']['strataDropped']}")
    for e in c["estimates"]:
        print(f"  {'OK ' if e['agrees'] else 'DIFF'} {e['term']}: coefficient {e['coefficient'][0]:.10g} / {e['coefficient'][1]:.10g}"
              f", se {e['se'][0]:.8g} / {e['se'][1]:.8g}")
print("\ntable keys:", report["table"]["keys"], "· shift named:", report["table"]["shiftNamed"])


## The result on record

First recorded on 8 November 2026, against release 0.5.0. Rerun on 26 November 2026, against release 0.9.0:
**Agrees**.

- **Same week:** 1,613 rows, 364 strata used, 16 dropped. Temperature coefficient 0.0375.
- **14 days before:** 1,586 rows, 363 strata used, 19 dropped. Temperature coefficient 0.0527.
- **28 days before:** 1,581 rows, 364 strata used, 18 dropped. Temperature coefficient 0.0198.
- **14 days after:** 1,583 rows, 363 strata used, 19 dropped. Temperature coefficient 0.0412.

Each count is the same on both sides. Only 1,499 rows are complete at every lag, so each model did use its own rows.
Every coefficient and standard error, for temperature and rainfall, is inside the criteria. The table names each
model's shift.

Running the cell above repeats the check on the same data.

## What this does not show

- That these lags, or one model per lag, are the right analysis.
- That the intervals are right for weekly data. They allow nothing for one week being like the next, and none is
  adjusted for the number of lags.
- That the dataset's temperature and rainfall measure what a mosquito meets. The temperature is InfoDengue's. The
  rainfall is CHIRPS at one grid cell.

In [ ]:
for item in report["notEstablished"]:
    print("-", item)
print()
print("fixture     :", report["fixture"]["id"], report["fixture"]["sha256"][:16])
print("statsmodels :", report["environment"]["statsmodels"])
print("runtime     :", report["environment"]["runtime"])
